<a href="https://colab.research.google.com/github/Shrutikraj/Deepfake_voice_video_Detection/blob/main/Model%20with%20modules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive #Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp /content/drive/MyDrive/deepfake_models/* /content/  #Copy all saved models into your session

In [ ]:
import os #Verifing files exist
os.listdir("/content")

['.config',
 'drive',
 'audio_mfcc_scaler.pkl',
 'audio_mfcc_classifier_baseline.pkl',
 'video_cnn_baseline.h5',
 'sample_data']

In [ ]:
import joblib #load  audio model

audio_model = joblib.load("audio_mfcc_classifier_baseline.pkl")
scaler = joblib.load("audio_mfcc_scaler.pkl")

print("Audio model + scaler loaded ✔")



Audio model + scaler loaded ✔


In [ ]:
from tensorflow.keras.models import load_model #loading video models

video_model = load_model("video_cnn_baseline.h5")

print("Video CNN model loaded ✔")


Video CNN model loaded ✔


In [ ]:
print(type(audio_model))
print(type(video_model))


<class 'sklearn.linear_model._logistic.LogisticRegression'>
<class 'keras.src.models.sequential.Sequential'>


In [ ]:
def predict_audio_authenticity(video_path):  #prediction function for audio

    audio_path = extract_audio_from_video(video_path)
    features = extract_mfcc_from_audio(audio_path)

    features_scaled = scaler.transform(features)

    pred = audio_model.predict(features_scaled)[0]
    prob = audio_model.predict_proba(features_scaled)[0]

    label_map = {
        0: "REAL AUDIO",
        1: "FAKE / SYNTHETIC AUDIO"
    }

    return label_map[pred], prob


In [ ]:
def predict_video_authenticity(video_path):  #prediction function for video

    frame = extract_center_frame(video_path)

    prob = video_model.predict(frame)[0][0]

    if prob >= 0.5:
        label = "FAKE VIDEO"
    else:
        label = "REAL VIDEO"

    return label, prob


In [ ]:
def run_multimodal_detector(video_path):  #Predicting audio & video together

    result = {}

    # AUDIO prediction
    audio_label, audio_prob = predict_audio_authenticity(video_path)
    result["audio_label"] = audio_label
    result["audio_confidence"] = float(max(audio_prob))

    # VIDEO prediction
    video_label, video_prob = predict_video_authenticity(video_path)
    result["video_label"] = video_label
    result["video_confidence"] = float(video_prob)

    # Final interpretation rule (basic for now)
    if audio_label == "REAL AUDIO" and video_label == "REAL VIDEO":
        result["final_case"] = "REAL VIDEO WITH REAL AUDIO"

    elif audio_label == "FAKE / SYNTHETIC AUDIO" and video_label == "REAL VIDEO":
        result["final_case"] = "REAL VIDEO BUT FAKE / DUBBED AUDIO"

    elif audio_label == "REAL AUDIO" and video_label == "FAKE VIDEO":
        result["final_case"] = "FAKE / ALTERED VIDEO WITH REAL AUDIO"

    else:
        result["final_case"] = "FULLY SYNTHETIC / DEEPFAKE CONTENT"

    return result


In [ ]:
import numpy as np
import cv2
import librosa
from moviepy.editor import VideoFileClip
from tensorflow.keras.models import load_model
import joblib


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [ ]:
!pip install moviepy librosa #Install moviepy & librosa


In [ ]:
!pip install librosa

In [ ]:
from moviepy.editor import VideoFileClip #Audio extraction from video

def extract_audio_from_video(video_path, output_path="temp_audio.wav"):

    video = VideoFileClip(video_path)

    audio = video.audio
    audio.write_audiofile(output_path, verbose=False, logger=None)

    return output_path


In [ ]:
def extract_mfcc_from_audio(audio_path, n_mfcc=40, segments=6):  #Audio to mfccs function

    y, sr = librosa.load(audio_path, sr=16000)

    total_len = len(y)
    segment_len = total_len // segments

    mfcc_features = []

    for i in range(segments):

        start = i * segment_len
        end = start + segment_len

        segment = y[start:end]

        if len(segment) == 0:
            continue

        mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=n_mfcc)

        mfcc_mean = np.mean(mfcc.T, axis=0)

        mfcc_features.extend(mfcc_mean)

    mfcc_features = np.array(mfcc_features).reshape(1, -1)

    return mfcc_features


In [ ]:
import cv2   #Extract center frame from video

def extract_center_frame(video_path):

    cap = cv2.VideoCapture(video_path)

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    target = frame_count // 2

    cap.set(cv2.CAP_PROP_POS_FRAMES, target)
    ret, frame = cap.read()

    cap.release()

    if not ret:
        raise ValueError("Unable to read video frame")

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (128, 128))

    frame = frame / 255.0
    frame = np.expand_dims(frame, axis=0)

    return frame


In [ ]:
video_path = "/content/download.mp4"   # Testing file sample

result = run_multimodal_detector(video_path)

for k,v in result.items():
    print(k,":",v)


OSError: MoviePy error: the file /content/download.mp4 could not be found!
Please check that you entered the correct path.

lip sync

In [ ]:
!pip install librosa opencv-python  #install dependancies


In [ ]:
import cv2   #Mouth-motion signal extractor (OpenCV)   This produces a time-series of mouth movement intensity.
import numpy as np

mouth_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_smile.xml"
)

def extract_mouth_motion_signal(video_path, sample_frames=40):

    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count == 0:
        return None

    step = max(frame_count // sample_frames, 1)
    motion_values = []

    idx = 0

    while idx < frame_count:

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        mouths = mouth_cascade.detectMultiScale(
            gray,
            scaleFactor=1.3,
            minNeighbors=15
        )

        if len(mouths) > 0:

            (x, y, w, h) = mouths[0]

            mouth_roi = gray[y:y+h, x:x+w]

            motion_energy = np.std(mouth_roi)

            motion_values.append(motion_energy)

        idx += step

    cap.release()

    return np.array(motion_values)


In [ ]:
import librosa  #Compute audio speech-activity signal

def extract_audio_activity_signal(video_path):

    audio_path = extract_audio_from_video(video_path)

    y, sr = librosa.load(audio_path, sr=16000)

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)

    onset_env = (onset_env - onset_env.mean()) / (onset_env.std() + 1e-6)

    return onset_env


In [ ]:
from scipy.signal import resample   #Compute Lip-Sync Correlation Score

def compute_lipsync_score(video_path):

    mouth_sig = extract_mouth_motion_signal(video_path)
    audio_sig = extract_audio_activity_signal(video_path)

    if mouth_sig is None or len(mouth_sig) < 3:
        return None, "Mouth not detected consistently"

    N = min(len(mouth_sig), len(audio_sig))

    if N < 3:
        return None, "Signals too short"

    mouth_sig = resample(mouth_sig, N)
    audio_sig = resample(audio_sig, N)

    mouth_sig = (mouth_sig - mouth_sig.mean()) / (mouth_sig.std() + 1e-6)
    audio_sig = (audio_sig - audio_sig.mean()) / (audio_sig.std() + 1e-6)

    score = np.corrcoef(mouth_sig, audio_sig)[0, 1]

    return score, "OK"


In [ ]:
def evaluate_lipsync(video_path): #convert score to label output

    score, status = compute_lipsync_score(video_path)

    if score is None:
        return "UNABLE TO ASSESS", None

    if score >= 0.65:
        label = "LIP-SYNC CONSISTENT"

    elif score >= 0.35:
        label = "PARTIAL / WEAK SYNC — POSSIBLE MANIPULATION"

    else:
        label = "LIP-SYNC MISMATCH — HIGH DUBBING RISK"

    return label, float(score)


In [ ]:
def explain_audio_result(prob):   #adding audio explaination function

    fake_prob = prob[1]
    real_prob = prob[0]

    if fake_prob > 0.85:
        return "Audio shows MFCC instability, formant drift and spectral artifacts — strongly suggesting synthetic / cloned speech."

    elif fake_prob > 0.60:
        return "Audio contains weak natural speech patterns with noticeable digital distortions — possible AI-generated or post-processed voice."

    elif real_prob > 0.85:
        return "Audio contains consistent MFCC structure, natural pitch variation and stable formant transitions — likely real human speech."

    else:
        return "Audio contains mixed natural and synthetic traits — authenticity uncertain."


In [ ]:
def explain_video_result(prob): #Adding video cnn explaination

    if prob > 0.85:
        return "Video CNN detected face-region texture inconsistency, boundary warping and abnormal compression — strong deepfake indicators."

    elif prob > 0.60:
        return "Video shows mild spatial distortions and face-region artifacts — possibly manipulated."

    elif prob < 0.40:
        return "Face structure, lighting and texture continuity appear natural — likely real video."

    else:
        return "Video contains weak or ambiguous manipulation signals."


In [ ]:
def explain_lipsync(score): #lip sync explaination

    if score is None:
        return "Lip-sync could not be evaluated due to insufficient mouth motion."

    if score >= 0.65:
        return "Mouth motion strongly aligns with speech rhythm — lip-sync consistent with natural talking behavior."

    elif score >= 0.35:
        return "Speech and mouth motion alignment is weak — possible editing, timing shift or partial manipulation."

    else:
        return "Mouth motion does not match speech timing — strong indication of dubbed or mismatched audio."


In [ ]:
def generate_explanation(result): #wrap everything into one explaination generator

    explanation = "\nDETAILED ANALYSIS REPORT\n\n"

    explanation += "AUDIO ANALYSIS:\n"
    explanation += "- " + explain_audio_result(result["audio_prob"]) + "\n\n"

    explanation += "VIDEO ANALYSIS:\n"
    explanation += "- " + explain_video_result(result["video_confidence"]) + "\n\n"

    explanation += "LIP-SYNC ANALYSIS:\n"
    explanation += "- " + explain_lipsync(result["lipsync_score"]) + "\n\n"

    explanation += "FINAL INTERPRETATION:\n"
    explanation += "- " + result["final_case"]

    return explanation


In [ ]:
def run_multimodal_detector(video_path):

    result = {}

    # --- AUDIO ---
    audio_label, audio_prob = predict_audio_authenticity(video_path)
    result["audio_label"] = audio_label
    result["audio_confidence"] = float(max(audio_prob))
    result["audio_prob"] = audio_prob

    # --- VIDEO ---
    video_label, video_prob = predict_video_authenticity(video_path)
    result["video_label"] = video_label
    result["video_confidence"] = float(video_prob)

    # --- LIP-SYNC ---
    lipsync_label, lipsync_score = evaluate_lipsync(video_path)
    result["lipsync_label"] = lipsync_label
    result["lipsync_score"] = lipsync_score

    # --- FINAL CASE INTERPRETATION ---
    if audio_label == "REAL AUDIO" and video_label == "REAL VIDEO":
        result["final_case"] = "REAL VIDEO WITH REAL AUDIO"

    elif audio_label == "FAKE / SYNTHETIC AUDIO" and video_label == "REAL VIDEO":
        result["final_case"] = "REAL VIDEO BUT FAKE / DUBBED AUDIO"

    elif audio_label == "REAL AUDIO" and video_label == "FAKE VIDEO":
        result["final_case"] = "FAKE / MANIPULATED VIDEO WITH REAL AUDIO"

    else:
        result["final_case"] = "FULLY SYNTHETIC / DEEPFAKE CONTENT"

    return result


In [ ]:
result = run_multimodal_detector(video_path)  #testing on same video

for k,v in result.items():
    print(k,":",v)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
audio_label : FAKE / SYNTHETIC AUDIO
audio_confidence : 1.0
audio_prob : [0. 1.]
video_label : FAKE VIDEO
video_confidence : 0.9774704575538635
lipsync_label : LIP-SYNC MISMATCH — HIGH DUBBING RISK
lipsync_score : -0.05637719859763128
final_case : FULLY SYNTHETIC / DEEPFAKE CONTENT


In [ ]:
!pip install -q openai-whisper


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 2.1 MB/s eta 0:00:00


In [ ]:
import whisper
stt_model = whisper.load_model("small")
print("Whisper ready ✔")


100%|████████████████████████████████████████| 461M/461M [00:03<00:00, 131MiB/s]


Whisper ready ✔


In [ ]:
def speech_to_text(audio_path):

    result = stt_model.transcribe(audio_path)

    transcript = result.get("text", "").strip()

    # Safe confidence estimation fallback
    conf = result.get("confidence", None)

    if conf is None:
        # Rough heuristic (based on word count)
        words = len(transcript.split())
        conf = min(1.0, max(0.3, words / 10))

    return transcript, float(conf)


In [ ]:
import re #Adding Semantic Consistency Check  This does NOT judge truthfulness —
                                              #only linguistic naturalness.

def semantic_consistency_check(transcript):

    text = transcript.lower()

    anomalies = 0

    if "uh" in text or "um" in text:
        anomalies += 1

    if len(text.split()) < 4:
        anomalies += 1

    if re.search(r"(repeated|loop|pattern)", text):
        anomalies += 1

    score = max(0, 1 - (anomalies * 0.25))

    if score > 0.75:
        status = "SEMANTICALLY NATURAL"
    elif score > 0.45:
        status = "SEMANTICALLY UNCERTAIN"
    else:
        status = "SEMANTICALLY SUSPICIOUS"

    return status, float(score)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity #Function to compute MFCC embedding
import librosa
import numpy as np

def compute_mfcc_embedding(audio_path):

    y, sr = librosa.load(audio_path, sr=16000)

    # --- remove silence ---
    y, _ = librosa.effects.trim(y, top_db=25)

    # --- extract MFCC ---
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)

    # --- normalize each MFCC feature ---
    mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) / \
           (mfcc.std(axis=1, keepdims=True) + 1e-6)

    # --- use BOTH mean and std as embedding ---
    emb_mean = np.mean(mfcc, axis=1)
    emb_std  = np.std(mfcc, axis=1)

    emb = np.concatenate([emb_mean, emb_std])

    # --- L2 normalize final embedding ---
    emb = emb / np.linalg.norm(emb)

    return emb.reshape(1, -1)



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def speaker_similarity(test_emb, reference_embeddings):

    sims = cosine_similarity(test_emb, reference_embeddings)

    avg_sim = float(np.mean(sims))

    # classification AFTER computing score
    if avg_sim >= 0.75:
        status = "VOICE MATCHES REFERENCE SPEAKER"
    elif avg_sim >= 0.55:
        status = "PARTIAL SPEAKER SIMILARITY"
    else:
        status = "VOICE DOES NOT MATCH REFERENCE SPEAKER"

    return status, avg_sim


In [ ]:
import glob #Load Reference Real-Speaker Samples

ref_paths = glob.glob("/content/shruti1.mp3")

reference_embeddings = [
    compute_mfcc_embedding(p) for p in ref_paths
]

reference_embeddings = np.vstack(reference_embeddings)

print("Loaded reference speaker samples ✔")


Loaded reference speaker samples ✔


In [ ]:
def extend_with_extra_modules(result, audio_path): #Attach Extensions to Pipeline

    # --- SPEECH TO TEXT ---
    transcript, stt_conf = speech_to_text(audio_path)
    result["transcript"] = transcript
    result["stt_confidence"] = stt_conf

    # --- SEMANTIC CHECK ---
    sem_status, sem_score = semantic_consistency_check(transcript)
    result["semantic_status"] = sem_status
    result["semantic_score"] = sem_score

    # --- SPEAKER SIMILARITY ---
    test_emb = compute_mfcc_embedding(audio_path)

    spk_status, spk_score = speaker_similarity(
        test_emb, reference_embeddings
    )

    result["speaker_match_status"] = spk_status
    result["speaker_similarity"] = spk_score

    return result


In [ ]:
import librosa #Convert MP3 → WAV
import soundfile as sf

y, sr = librosa.load("/content/shruti1.mp3", sr=16000)
sf.write("/content/shruti1.mp3", y, sr)

print("Converted to WAV ✔")


Converted to WAV ✔


In [ ]:

emb = compute_mfcc_embedding("/content/shruti1.mp3") #Build embedding and compare with reference speakers


In [ ]:
status, score = speaker_similarity(emb, reference_embeddings)

print("Speaker Match Status:", status)
print("Similarity Score:", score)


Speaker Match Status: VOICE MATCHES REFERENCE SPEAKER
Similarity Score: 1.0


In [ ]:
transcript, conf = speech_to_text("/content/shruti1.mp3")
print("Transcript:", transcript)
print("STT Confidence:", conf)

sem_status, sem_score = semantic_consistency_check(transcript)
print("Semantic Status:", sem_status)
print("Semantic Score:", sem_score)


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Transcript: It's meant for basic voice check. I will continue to speak naturally so the audio reflects my regular voice. This recording helps test clarity, pitch and pacing. The sound serve as a clean and useful 20 seconds.
STT Confidence: 1.0
Semantic Status: SEMANTICALLY NATURAL
Semantic Score: 1.0


In [ ]:
for k,v in result.items(): #Print extended results
    print(k,":",v)


audio_label : FAKE / SYNTHETIC AUDIO
audio_confidence : 1.0
audio_prob : [0. 1.]
video_label : FAKE VIDEO
video_confidence : 0.9774704575538635
lipsync_label : LIP-SYNC MISMATCH — HIGH DUBBING RISK
lipsync_score : -0.05637719859763128
final_case : FULLY SYNTHETIC / DEEPFAKE CONTENT


not important to run this specifically as we already used all these function for output

video input


In [ ]:
result = run_multimodal_detector(video_path) #if input is video


if input is audio

In [ ]:
y, sr = librosa.load("shruti1.mp3", sr=16000)  #first convert audio to video
sf.write("shruti1.wav", y, sr)


In [ ]:
transcript, conf = speech_to_text("shruti1.wav") #speech to text


In [ ]:
semantic_status, semantic_score = semantic_consistency_check(transcript) #semantic analysis


In [ ]:
emb = compute_mfcc_embedding("shruti1.wav") #speaker similarity
status, score = speaker_similarity(emb, reference_embeddings)


In [ ]:
print("Number of reference files:", len(ref_paths))
print("Unique reference paths:", len(set(ref_paths)))

print("Reference embedding shape:", reference_embeddings.shape)

print("First 5 paths:")
for p in ref_paths[:5]:
    print(" -", p)


In [ ]:
import numpy as np

print("Embedding variance:", np.var(reference_embeddings))
print("Unique rows:", np.unique(reference_embeddings.round(4), axis=0).shape[0])
